# Read BigLake Iceberg Table with PySpark

This notebook reads from the BigLake Iceberg REST Catalog table:
`httparchive_lakehouse_us.sample_data.pages_10k` (stored in `gs://httparchive_lakehouse_us/`).

## Google Cloud Authentication
- Authenticates Google Cloud on Colab via `auth.authenticate_user()`.
- Retrieve active Google Cloud credentials and refresh the OAuth2 access token for catalog authorization.

In [ ]:
import sys

import google.auth
import google.auth.transport.requests

if "google.colab" in sys.modules:
    from google.colab import auth
    auth.authenticate_user(project_id="httparchive")

credentials, project_id = google.auth.default(
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)
credentials.refresh(google.auth.transport.requests.Request())

## Optional Java Setup & Initializing SparkSession with BigLake REST Catalog
Configures:
- Detects runtime environment and installs `pyspark` if missing.
- Dynamic Iceberg runtime selection
- BigLake REST Catalog endpoint (`https://biglake.googleapis.com/iceberg/v1/restcatalog`)
- Access delegation via `vended-credentials` for secure, direct GCS access
- 4GB driver memory to handle metadata processing smoothly

In [ ]:
import os
import subprocess

from pyspark.sql import SparkSession


if "JAVA_HOME" not in os.environ:
    JAVA_PATH = "/opt/homebrew/opt/openjdk@17"
    if os.path.exists(JAVA_PATH):
        os.environ["JAVA_HOME"] = JAVA_PATH

try:
    import pyspark
except ImportError:
    print("PySpark not found. Installing PySpark...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark"])
    import pyspark

catalog_name = "httparchive_lakehouse_us"

spark = (
    SparkSession.builder
    .appName("BigLake-Iceberg-Local-Read")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.jars.packages", (
        "org.apache.iceberg:iceberg-spark-runtime-4.0_2.13:1.11.0,"
        "org.apache.iceberg:iceberg-gcp-bundle:1.11.0"
        )
    )
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
    )
    .config("spark.sql.defaultCatalog", catalog_name)
    .config(f"spark.sql.catalog.{catalog_name}", "org.apache.iceberg.spark.SparkCatalog")
    .config(f"spark.sql.catalog.{catalog_name}.type", "rest")
    .config(f"spark.sql.catalog.{catalog_name}.uri", "https://biglake.googleapis.com/iceberg/v1/restcatalog")
    .config(f"spark.sql.catalog.{catalog_name}.warehouse", f"bl://projects/httparchive/catalogs/{catalog_name}")
    .config(f"spark.sql.catalog.{catalog_name}.io-impl", "org.apache.iceberg.gcp.gcs.GCSFileIO")
    .config(f"spark.sql.catalog.{catalog_name}.header.x-goog-user-project", "httparchive")
    .config(f"spark.sql.catalog.{catalog_name}.header.X-Iceberg-Access-Delegation", "vended-credentials")
    .config(f"spark.sql.catalog.{catalog_name}.token", credentials.token)
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
print("SparkSession initialized successfully!")

## Load Iceberg Table & Inspect Schema

In [ ]:
df = spark.table(f"`{catalog_name}`.sample_data.pages_10k")
df.printSchema()
print(f"Total rows: {df.count():,}")

## Partition & Cluster Breakdown


In [ ]:
df.groupBy("date", "client", "is_root_page", "rank") \
  .count() \
  .orderBy("date", "client", "is_root_page", "rank") \
  .show()


## Sample Records

In [ ]:
df.select("date", "client", "rank", "page", "is_root_page") \
  .orderBy("rank") \
  .show(10, truncate=False)

## Query Stringified JSON Fields

In [ ]:
from pyspark.sql import functions as F

df.select(
    "client",
    "rank",
    "page",
    F.get_json_object("summary", "$.bytesTotal").cast("long").alias("bytes_total"),
    F.get_json_object("summary", "$.reqTotal").cast("int").alias("req_total"),
    F.get_json_object("summary", "$.bytesJs").cast("long").alias("bytes_js"),
    F.get_json_object("summary", "$.bytesImg").cast("long").alias("bytes_img")
).filter("bytes_total IS NOT NULL") \
 .orderBy("rank") \
 .show(5, truncate=False)